### Import Libraries

In [1]:
import mlflow
import os
from pathlib import Path
import pandas as pd
import joblib
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
import shap

In [2]:
mlflow.__version__

'3.5.1'

In [3]:
os.listdir("../models")

['Baseline_LogReg_Model.pkl',
 'Baseline_RandomForest_Model.pkl',
 'Tuned_RandomForest_Model.pkl']

### Initialize MLflow Tracking:

In [4]:
# Find the project root (Stock-analysis)
project_root = Path.cwd().parent
print("Project root:", project_root)

# Build the tracking URI pointing to Stock-analysis/mlruns
tracking_uri = "file:" + str(project_root / "mlruns")
print("Tracking URI:", tracking_uri)

Project root: c:\Users\estee\OneDrive\Desktop\Amdari\Project 1 - Stock_Mkt_Trend Analysis\Stock-analysis
Tracking URI: file:c:\Users\estee\OneDrive\Desktop\Amdari\Project 1 - Stock_Mkt_Trend Analysis\Stock-analysis\mlruns


In [5]:
# Tell MLflow to use that tracking folder
mlflow.set_tracking_uri(tracking_uri)

# Create/select the experiment
experiment_name = "Stock Market Trend Prediction"
mlflow.set_experiment(experiment_name)

print("Experiment Name set to:", experiment_name)


2025/11/29 18:58:29 INFO mlflow.tracking.fluent: Experiment with name 'Stock Market Trend Prediction' does not exist. Creating a new experiment.


Experiment Name set to: Stock Market Trend Prediction


### Log Phase 1 Logistic Regression Baseline Model

In [6]:
# Load the modeling dataset
df = pd.read_csv('../data/modeling_dataset.csv')

df['date'] = pd.to_datetime(df['date'])

# Temporal Split: 70% Train, 30% Test
split_date = df['date'].quantile(0.7)
train_df = df[df['date'] <= split_date].copy()
test_df  = df[df['date'] > split_date].copy()

# Identify the needed variables
num_cols = df.select_dtypes(include=['int', 'float']).columns
target_col = 'trend_label'

# Data Splitting
X_test = test_df[num_cols]
y_test = test_df[target_col]

In [7]:
# LOAD THE SAVED BASELINE LOGISTIC REGRESSION MODEL
log_reg_model = joblib.load("../models/Baseline_LogReg_Model.pkl")

# Encode target variable
le = LabelEncoder()
le.fit(df['trend_label'])     

y_test_enc = le.transform(y_test)

# MAKE PREDICTIONS
y_pred = log_reg_model.predict(X_test)

# CALCULATE METRICS
acc  = accuracy_score(y_test_enc, y_pred)
prec = precision_score(y_test_enc, y_pred, average="weighted")
rec  = recall_score(y_test_enc, y_pred, average="weighted")
f1   = f1_score(y_test_enc, y_pred, average="weighted")

print("Baseline Logistic Regression:")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)

Baseline Logistic Regression:
Accuracy: 0.340301724137931
Precision: 0.11580526345124852
Recall: 0.340301724137931
F1-score: 0.17280476681582027


Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [8]:
# Plot confusion Matrix

cm = confusion_matrix(y_test_enc, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Baseline Logistic Regression")

plt.tight_layout()
plt.savefig("baseline_logreg_confusion_matrix.png", dpi=300)
plt.close()


In [9]:
# Plot Metrics by class

# To generate classification report as a dictionary
report_dict = classification_report(y_test_enc, y_pred, output_dict=True)
df_report = pd.DataFrame(report_dict).T

# Extract only classes 0, 1, 2
df_scores = df_report.loc[['0', '1', '2'], ['precision', 'recall', 'f1-score']]
df_scores

# To plt grouped barchart

# Data
metrics = ['precision', 'recall', 'f1-score']
classes = df_scores.index.tolist()
values = df_scores.values

# Bar positions
x = np.arange(len(classes))   # positions for classes: 0, 1, 2
width = 0.25                  # width of each bar

plt.figure(figsize=(10, 6))

# Plot bars
plt.bar(x - width, values[:, 0], width, label='Precision')
plt.bar(x,         values[:, 1], width, label='Recall')
plt.bar(x + width, values[:, 2], width, label='F1-score')

# Labels and title
class_labels = ['Downtrend', 'Sideways', 'Uptrend']

plt.xlabel("Classes")
plt.ylabel("Scores")
plt.title("LogisticRegression: Precision, Recall, and F1-score by Class")
plt.xticks(x, class_labels)
plt.legend()

plt.tight_layout()
plt.savefig("baseline_logreg_class_scores.png", dpi=300)
plt.close()

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


### MLflow Log

In [10]:
# Log to MLflow
mlflow.set_experiment("Stock Market Trend Prediction")

with mlflow.start_run(run_name="Baseline_Logistic_Regression"):

    # Tag as baseline model
    mlflow.set_tag("model_role", "baseline_model")
    mlflow.set_tag("model_family", "LogisticRegression")

    # Log required parameters
    params = log_reg_model.get_params()
    mlflow.log_param("max_iter", params.get("max_iter"))
    mlflow.log_param("random_state", params.get("random_state"))

    # Log required metrics (names match the spec)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    # log plots as artifacts
    mlflow.log_artifact("baseline_logreg_confusion_matrix.png")
    mlflow.log_artifact("baseline_logreg_class_scores.png")

    # Log the model
    mlflow.sklearn.log_model(log_reg_model, "model")

print("Baseline Logistic Regression logged to MLflow")

2025/11/29 18:58:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/29 18:58:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Baseline Logistic Regression logged to MLflow


### Log Phase 1 Random Forest Baseline Model

In [11]:
# LOAD THE SAVED BASELINE RANDOM FOREST MODEL
rf_model = joblib.load("../models/Baseline_RandomForest_Model.pkl")

# MAKE PREDICTIONS
y_pred_rf = rf_model.predict(X_test)

# CALCULATE METRICS (use encoded labels)
acc_rf  = accuracy_score(y_test_enc, y_pred_rf)
prec_rf = precision_score(y_test_enc, y_pred_rf, average="weighted")
rec_rf  = recall_score(y_test_enc, y_pred_rf, average="weighted")
f1_rf   = f1_score(y_test_enc, y_pred_rf, average="weighted")

print("Baseline Random Forest:")
print("  Accuracy :", acc_rf)
print("  Precision:", prec_rf)
print("  Recall   :", rec_rf)
print("  F1-score :", f1_rf)

Baseline Random Forest:
  Accuracy : 0.36810344827586206
  Precision: 0.3500182191354212
  Recall   : 0.36810344827586206
  F1-score : 0.20840107506905498


Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [12]:
# To save confusion matrix as artifact

cm_rf = confusion_matrix(y_test_enc, y_pred_rf)

plt.figure(figsize=(7,5))
sns.heatmap(cm_rf, annot=True, fmt="d",
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap="Blues")

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.savefig("baseline_rf_confusion_matrix.png", dpi=300)
plt.close()

In [13]:
# To add class plot as an artifact

# To generate classification report as a dictionary
report_dict_rf = classification_report(y_test_enc, y_pred_rf, output_dict=True)
df_report_rf = pd.DataFrame(report_dict_rf).T

# Extract only classes 0, 1, 2
df_scores_rf = df_report_rf.loc[['0', '1', '2'], ['precision', 'recall', 'f1-score']]
df_scores_rf

# To plt grouped barchart

# Data
metrics_rf = ['precision', 'recall', 'f1-score']
classes_rf = df_scores_rf.index.tolist()
values_rf = df_scores_rf.values

# Bar positions
x = np.arange(len(classes_rf))   
width = 0.25                 

plt.figure(figsize=(10, 6))

# Plot bars
plt.bar(x - width, values[:, 0], width, label='Precision')
plt.bar(x,         values[:, 1], width, label='Recall')
plt.bar(x + width, values[:, 2], width, label='F1-score')

# Labels and title
class_labels_rf = ['Downtrend', 'Sideways', 'Uptrend']

plt.xlabel("Classes")
plt.ylabel("Scores")
plt.title("RandomForest: Precision, Recall, and F1-score by Class")
plt.xticks(x, class_labels_rf)
plt.legend()

plt.tight_layout()
plt.savefig("baseline_rf_class_scores.png", dpi=300)
plt.close()

### MLflow RF

In [14]:
mlflow.set_experiment("Stock Market Trend Prediction")

with mlflow.start_run(run_name="Baseline_RandomForest"):

    # Tags
    mlflow.set_tag("model_role", "baseline_model")
    mlflow.set_tag("model_family", "RandomForestClassifier")

    # Parameters
    rf_params = rf_model.get_params()
    mlflow.log_param("n_estimators",    rf_params.get("n_estimators"))
    mlflow.log_param("max_depth",       rf_params.get("max_depth"))
    mlflow.log_param("min_samples_split", rf_params.get("min_samples_split"))

    # Metrics
    mlflow.log_metric("accuracy",  acc_rf)
    mlflow.log_metric("precision", prec_rf)
    mlflow.log_metric("recall",    rec_rf)
    mlflow.log_metric("f1_score",  f1_rf)

     # log plots as artifacts
    mlflow.log_artifact("baseline_rf_confusion_matrix.png")
    mlflow.log_artifact("baseline_rf_class_scores.png")


    # Log the model itself
    input_example = X_test.iloc[:1]
    mlflow.sklearn.log_model(rf_model, "model", input_example=input_example)

print("Baseline Random Forest logged to MLflow")


2025/11/29 18:58:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.


Baseline Random Forest logged to MLflow


### Log Phase 1 Tuned Random Forest with Feature Importance

In [15]:
df_tuned = pd.read_csv("../data/final_dataset.csv")

# Define variables
target = 'trend_label'
features = [col for col in df_tuned.columns if col not in ['ticker', 'date', target]]

# Temporal Data Split
df_tuned['date'] = pd.to_datetime(df_tuned['date'])

df_split_date = df_tuned['date'].quantile(0.7)
train_data = df_tuned[df_tuned['date'] <= df_split_date].copy()
test_data = df_tuned[df_tuned['date'] > df_split_date].copy()

# Data Splitting
train_X = train_data[features]
train_y = train_data[target]
test_X  = test_data[features]
test_y  = test_data[target]

# Load the saved tuned random forest model
rf_tuned = joblib.load("../models/Tuned_RandomForest_Model.pkl")

# Encode target variable
le_rf = LabelEncoder()
y_train_tuned = le_rf.fit_transform(train_y)
y_test_tuned = le_rf.transform(test_y)

# Make Prediction
para_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 20, 40, None],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rs = RandomizedSearchCV(
    rf_tuned,
    param_distributions=para_grid,
    n_iter=20,
    cv=3,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1
)

rs.fit(train_X, y_train_tuned)
best_model = rs.best_estimator_
pred_y= best_model.predict(test_X)

# CALCULATE METRICS
acc_tuned = accuracy_score(y_test_tuned, pred_y)
prec_tuned = precision_score(y_test_tuned, pred_y, average="weighted")
rec_tuned = recall_score(y_test_tuned, pred_y, average="weighted")
f1_tuned = f1_score(y_test_tuned, pred_y, average="weighted")

print("Feature Importance Tuned Random Forest:")
print("Accuracy:", acc_tuned)
print("Precision:", prec_tuned)
print("Recall:", rec_tuned)
print('F1 Score:', f1_tuned)

Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


Feature Importance Tuned Random Forest:
Accuracy: 0.35064655172413794
Precision: 0.34401872338151385
Recall: 0.35064655172413794
F1 Score: 0.34055863446267537


### Add Artifacts

In [16]:
# Add Feature Importance

importance = best_model.feature_importances_
feat_names = test_X.columns

fi_df = pd.DataFrame({
    'feature': feat_names,
    'importance': importance
}).sort_values(by='importance', ascending=False)

# Graphical Representation of Feature Importance

plt.figure(figsize=(10,6))
fi_df.head(10).plot(kind='barh', x='feature', y='importance')
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance Score")

plt.tight_layout()
plt.savefig("featureImportance_random_forest.png", dpi=300)
plt.close()

<Figure size 1000x600 with 0 Axes>

In [17]:
# Load confusion matrix

cm_tuned = confusion_matrix(y_test_tuned, pred_y)

plt.figure(figsize=(6,4))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_rf.classes_, yticklabels=le_rf.classes_)
plt.title("Tuned Random Forest (FI) Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.savefig("featureImportance_rf_confusion_matrix.png", dpi=300)
plt.close()

In [18]:
# Load shap

class_names = le_rf.inverse_transform(best_model.classes_)
X_shap = test_X.copy()

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap)

shap.summary_plot(shap_values, X_shap, plot_type="bar", class_names=class_names, show=False)
plt.title("SHAP Feature Importance - Tuned Random Forest")

plt.tight_layout()
plt.savefig("shap_tuned_rf.png", dpi=300, bbox_inches="tight")
plt.close()

### MLflow Tuned Model

In [19]:
mlflow.set_experiment("Stock Market Trend Prediction")

with mlflow.start_run(run_name="Tuned_RandomForest"):

    # Tags
    mlflow.set_tag("model_role", "production_candidate")
    mlflow.set_tag("model_family", "RandomForestClassifier")

    # Parameters
    rf_parameters = rf_tuned.get_params()
    mlflow.log_param("n_estimators",    rf_parameters.get("n_estimators"))
    mlflow.log_param("max_depth",       rf_parameters.get("max_depth"))
    mlflow.log_param("min_samples_split", rf_parameters.get("min_samples_split"))

    # Metrics
    mlflow.log_metric("accuracy",  acc_tuned)
    mlflow.log_metric("precision", prec_tuned)
    mlflow.log_metric("recall",    rec_tuned)
    mlflow.log_metric("f1_score",  f1_tuned)

     # log plots as artifacts
    mlflow.log_artifact("featureImportance_random_forest.png")
    mlflow.log_artifact("featureImportance_rf_confusion_matrix.png")
    mlflow.log_artifact("shap_tuned_rf.png")


    # Log the model itself
    input_example = test_X.iloc[:1]
    mlflow.sklearn.log_model(best_model, "tuned_rf_model", input_example=input_example)

print("Tuned Random Forest (FI) logged to MLflow")

2025/11/29 19:26:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.


Tuned Random Forest (FI) logged to MLflow
